# Progress Review 2 model outputs
Run the scripts from START_HERE.txt first, or inspect the included results below. This notebook does not retrain automatically.

In [1]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
def display(value):
    print(value.to_string() if hasattr(value, "to_string") else value)
ROOT = Path.cwd()
if not (ROOT / 'run_all.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'run_all.py').exists(), 'Open this notebook from the extracted project folder.'
sys.path.insert(0, str(ROOT))


## Dataset and features

In [2]:
audit = json.loads((ROOT / 'results/data_audit.json').read_text())
display(pd.Series({k: audit[k] for k in ['cohort_SKUs', 'complete_weeks', 'model_rows', 'rows_by_split']}))
display(pd.read_csv(ROOT / 'data/processed/model_panel.csv').head())

cohort_SKUs                                                    5615
complete_weeks                                                   12
model_rows                                                    44920
rows_by_split     {'train': 22460, 'validation': 11230, 'test': ...
           SKU  week_start  units category size first_observed_date  lag_1  lag_2  lag_3  lag_4  mean_4     std_4  zero_weeks_4  change_1 history_available_through       split
0  AN201-RED-M  2022-05-02      0   BOTTOM    M          2022-04-07    1.0    0.0    0.0    1.0    0.50  0.500000             2       1.0                2022-05-01       train
1  AN201-RED-M  2022-05-09      0   BOTTOM    M          2022-04-07    0.0    1.0    0.0    0.0    0.25  0.433013             3      -1.0                2022-05-08       train
2  AN201-RED-M  2022-05-16      0   BOTTOM    M          2022-04-07    0.0    0.0    1.0    0.0    0.25  0.433013             3       0.0                2022-05-15       train
3  AN201-RED-M  2022-05-

## Validation and final test results

In [3]:
display(pd.read_csv(ROOT / 'results/model_comparison.csv'))
print(json.loads((ROOT / 'results/selection.json').read_text())['selected_model'])

         stage                   model       MAE      RMSE   WAPE_pct        R2  mean_error_pred_minus_actual   rows  actual_total  predicted_total
0   validation           NaiveLastWeek  1.103740  2.400115  77.237039  0.543951                     -0.102850  11230       16048.0     14893.000000
1   validation              Mean4Weeks  1.087667  2.261286  76.112288  0.595183                     -0.147952  11230       16048.0     14386.500000
2   validation                   Ridge  1.085382  2.372587  75.952394  0.554352                     -0.264662  11230       16048.0     13075.851064
3   validation     RandomForestHistory  1.109237  2.477297  77.621709  0.514148                     -0.274796  11230       16048.0     12962.039397
4   validation  RandomForestAttributes  1.107939  2.421009  77.530863  0.535976                     -0.270958  11230       16048.0     13005.139599
5   validation    HistGradientBoosting  1.113085  2.379383  77.890969  0.551796                     -0.257371  1

## Forecast errors and feature checks

In [4]:
display(pd.read_csv(ROOT / 'results/test_metrics_by_week.csv'))
display(pd.read_csv(ROOT / 'results/largest_test_errors.csv').head(10))

                     model  week_start       MAE      RMSE    WAPE_pct        R2  mean_error_pred_minus_actual  rows  actual_total  predicted_total
0     HistGradientBoosting  2022-06-13  1.005601  2.067254   84.565706  0.320058                      0.125609  5615        6677.0      7382.293608
1     HistGradientBoosting  2022-06-20  0.947194  1.773892   98.673375  0.509993                      0.267583  5615        5390.0      6892.479291
2               Mean4Weeks  2022-06-13  1.012066  2.211413   85.109331  0.221920                      0.125156  5615        6677.0      7379.750000
3               Mean4Weeks  2022-06-20  0.973419  2.015029  101.405380  0.367718                      0.367809  5615        5390.0      7455.250000
4            NaiveLastWeek  2022-06-13  1.141229  2.774711   95.971245 -0.224953                      0.280321  5615        6677.0      8251.000000
5            NaiveLastWeek  2022-06-20  0.954764  2.002625   99.461967  0.375478                      0.229207  

![Model comparison](../results/figures/test_model_comparison.png)
![Feature importance](../results/figures/validation_feature_importance.png)

## Historical forecast and assumed stock scenarios

In [5]:
display(pd.read_csv(ROOT / 'results/next_week_forecast.csv').head())
display(pd.read_csv(ROOT / 'results/reorder_scenarios.csv'))
from src.forecast import reorder_quantity
print('Illustrative reorder:', reorder_quantity(weekly_sales=10, on_hand=5, on_order=3, backorders=1, lead_time_days=7, review_period_days=7, safety_stock_units=2))

                SKU category size  week_start history_available_through  predicted_weekly_shipped_units  model                                  forecast_type
0       AN201-RED-M   BOTTOM    M  2022-06-27                2022-06-26                        0.007296  Ridge  historical_demonstration_not_current_forecast
1    AN202-ORANGE-M   BOTTOM    M  2022-06-27                2022-06-26                        0.007296  Ridge  historical_demonstration_not_current_forecast
2    AN202-ORANGE-S   BOTTOM    S  2022-06-27                2022-06-26                        0.000000  Ridge  historical_demonstration_not_current_forecast
3  AN202-ORANGE-XXL   BOTTOM  2XL  2022-06-27                2022-06-26                        0.000000  Ridge  historical_demonstration_not_current_forecast
4    AN205-YELLOW-S   BOTTOM    S  2022-06-27                2022-06-26                        0.000000  Ridge  historical_demonstration_not_current_forecast
               SKU       category size  week_start h

## Verification

In [6]:
verification = json.loads((ROOT / 'results/verification.json').read_text())
print('Passed:', verification['passed'], '| Checks:', verification['tests_run'])
print((ROOT / 'results/test_log.txt').read_text())

Passed: True | Checks: 29
test_bad_inputs_rejected (test_inventory.InventoryTests.test_bad_inputs_rejected) ... ok
test_batch_inputs_known_and_valid (test_inventory.InventoryTests.test_batch_inputs_known_and_valid) ... ok
test_comparison_uses_identical_inputs (test_inventory.InventoryTests.test_comparison_uses_identical_inputs) ... ok
test_hand_worked_receipt_timing (test_inventory.InventoryTests.test_hand_worked_receipt_timing) ... ok
test_long_lead_keeps_orders_in_position (test_inventory.InventoryTests.test_long_lead_keeps_orders_in_position) ... ok
test_saved_ledger_conservation_and_timing (test_inventory.InventoryTests.test_saved_ledger_conservation_and_timing) ... ok
test_sku_selection_ignores_validation_and_test_targets (test_inventory.InventoryTests.test_sku_selection_ignores_validation_and_test_targets) ... ok
test_stock_decisions_recompute (test_inventory.InventoryTests.test_stock_decisions_recompute) ... ok
test_stock_report_assets_and_guide_exist (test_inventory.InventoryTe

## Stock replenishment and exploratory simulation
The same policy compares a four-week usage average and the validation-selected model. All stock conditions are assumed; this is not observed lost sales or proof of savings. Read docs/GROUP_GUIDE.html.

In [7]:
display(pd.read_csv(ROOT / 'results/inventory_comparison.csv'))
display(pd.read_csv(ROOT / 'results/stock_decisions.csv').head(10))
display(pd.read_csv(ROOT / 'results/inventory_ledger.csv').head(8))

         scenario       model  sku_count  weeks  demand_proxy_units  fulfilled_units  unmet_proxy_units  simulated_fill_rate_pct  mean_end_stock_per_sku_week  order_events  ordered_units  final_stock_units  final_in_transit_units
0      zero_start  Mean4Weeks        100      2                1516              689                827                45.448549                       12.785           127           3335               2557                      89
1      zero_start       Ridge        100      2                1516              689                827                45.448549                        9.710           117           2712               1942                      81
2  one_week_start  Mean4Weeks        100      2                1516             1400                116                92.348285                       13.955           149           2267               1903                     579
3  one_week_start       Ridge        100      2                1516             